# Volume 6, Chapter 9: Lp Spaces (Python companion)

Exercises first, solutions at the end. $L^p$ on a computer: the shapes of unit balls and the behaviour of $p\mapsto\|f\|_p$, image blurring, sharpening and edge detection by convolution (direct and by FFT), approximate identities against a jump, and the sharpness of Hardy's inequality. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P9.1 (★) The shapes of $L^p$.** Plot the unit balls of $\|\cdot\|_p$ in the plane for several $p$, and compute $\|f\|_p$ as a function of $p$ for $f(x)=\frac1{\sqrt x}\mathbf 1_{]0,1[}$, for $f(x)=e^{-x}$ on $]0,\infty[$ and for a random step function on $[0,1]$. Check Exercise 9.4 ($\|f\|_p\to\|f\|_\infty$) and Exercise 9.3 (convexity of $\frac1p\mapsto\ln\|f\|_p$).

**P9.2 (★★) CS application: blurring and sharpening images.** Build a test image (shapes and text-like stripes) and convolve it with Gaussian kernels of increasing width, by direct summation and by the FFT. Then sharpen it by *unsharp masking*, $f+\alpha(f-f*G_t)$, and detect edges with a discrete Laplacian. Check Young's inequality $\|f*g\|_2\le\|f\|_2\|g\|_1$ on the result.

**P9.3 (★★) Approximate identities at work.** Convolve the indicator of $[-1,1]$ with Gaussian, Poisson and box kernels $\varphi_t$ for $t\to0$. Measure $\|f*\varphi_t-f\|_1$ and $\|f*\varphi_t-f\|_\infty$, and the value at the jump. Which convergence of Theorem 9.10 holds, and which cannot?

**P9.4 (★) Hardy's inequality is sharp.** Compute $\frac{\|F\|_p}{\|f\|_p}$ for $f=x^{-1/p}\mathbf 1_{[1,N]}$ and growing $N$, with $F(x)=\frac1x\int_0^xf$, and watch it approach $\frac p{p-1}$ (Exercise 9.5).

## Solutions

### P9.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
th = np.linspace(0, 2 * np.pi, 721)
plt.figure(figsize=(4.5, 4.5))
for p in [0.5, 1, 1.5, 2, 4, 10]:
    c, s = np.cos(th), np.sin(th); r = (np.abs(c) ** p + np.abs(s) ** p) ** (-1 / p)
    plt.plot(r * c, r * s, label=f"p={p}")
plt.plot([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], "k", lw=0.8, label="p=inf"); plt.gca().set_aspect("equal"); plt.legend(fontsize=7); plt.show()

from scipy.integrate import quad
def norm(f, a, b, p): return quad(lambda x: abs(f(x)) ** p, a, b, limit=200)[0] ** (1 / p)
print("1/sqrt(x) on (0,1):", [(p, round(norm(lambda x: x ** -0.5, 0, 1, p), 4)) for p in [1, 1.5, 1.9, 1.99]], "-> infinite at p = 2")
ps = np.array([1, 2, 4, 8, 16, 32, 64, 128])
print("exp(-x) on (0,inf): ||f||_p = p^(-1/p):", [round(float(norm(lambda x: np.exp(-x), 0, np.inf, p)), 4) for p in ps], " -> 1 = ||f||_inf")
rng = np.random.default_rng(0); h = rng.random(20)                  # step function, 20 equal steps
def step_norm(p): return (np.mean(h ** p)) ** (1 / p)
print("random step function: max =", h.max().round(4), " norms:", [round(float(step_norm(p)), 4) for p in ps])
s = np.linspace(0.02, 1, 200); L = np.log([step_norm(1 / x) for x in s])
print("second differences of log ||f||_{1/s} (should be >= 0):", np.min(np.diff(L, 2)) >= -1e-12)

For $p<1$ the "unit ball" is not convex: $\|\cdot\|_p$ is then not a norm, which is why $L^p$ theory starts at $p=1$. The norms of $\frac1{\sqrt x}$ on $]0,1[$ are $\bigl(\frac2{2-p}\bigr)^{1/p}$, finite for $p<2$ and exploding as $p\to2$. For $e^{-x}$ on $]0,\infty[$ (an infinite measure), $\|f\|_p=p^{-1/p}$ first decreases, from $1$ at $p=1$ to its minimum near $p=e$, then increases back to $1=\|f\|_\infty$: without a finite measure, $p\mapsto\|f\|_p$ need not be monotone; for the step function the norms increase towards the maximum value, as Exercise 9.4 predicts (on a probability space they increase with $p$, by Jensen). On a uniform grid in $s=\frac1p$, the second differences of $s\mapsto\ln\|f\|_{1/s}$ are non-negative: the convexity of Exercise 9.3.

### P9.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n = 256
y, x = np.mgrid[0:n, 0:n] / n
img = ((x - 0.3) ** 2 + (y - 0.35) ** 2 < 0.04).astype(float)                    # a disc
img += ((abs(x - 0.72) < 0.12) & (abs(y - 0.3) < 0.18)) * 0.7                     # a square
img += ((y > 0.65) & (y < 0.9) & (np.sin(60 * x) > 0)) * 0.9                       # stripes, like text
img = np.clip(img, 0, 1)

def gauss_kernel(sigma, size=None):
    size = size or int(6 * sigma) | 1
    r = np.arange(size) - size // 2
    g = np.exp(-r ** 2 / (2 * sigma ** 2)); g /= g.sum()
    return np.outer(g, g)

def conv_fft(f, k):
    K = np.zeros_like(f); s = k.shape[0]
    K[:s, :s] = k; K = np.roll(K, (-(s // 2), -(s // 2)), axis=(0, 1))          # centre the kernel at the origin
    return np.real(np.fft.ifft2(np.fft.fft2(f) * np.fft.fft2(K)))                # periodic convolution

def conv_direct(f, k):
    s = k.shape[0]; out = np.zeros_like(f)
    for i in range(s):
        for j in range(s):
            out += k[i, j] * np.roll(f, (i - s // 2, j - s // 2), axis=(0, 1))
    return out

k = gauss_kernel(2.0)
print("direct and FFT convolutions agree:", np.allclose(conv_fft(img, k), conv_direct(img, k)))
fig, ax = plt.subplots(1, 5, figsize=(14, 3))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("original")
for a, sg in zip(ax[1:3], [2, 6]):
    a.imshow(conv_fft(img, gauss_kernel(sg)), cmap="gray"); a.set_title(f"Gaussian blur, sigma={sg}")
blur = conv_fft(img, gauss_kernel(3)); sharp = img + 1.5 * (img - blur)
ax[3].imshow(np.clip(sharp, 0, 1), cmap="gray"); ax[3].set_title("unsharp masking")
lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float)
ax[4].imshow(np.abs(conv_fft(img, lap)), cmap="gray"); ax[4].set_title("|Laplacian|: edges")
for a in ax: a.axis("off")
plt.show()
for sg in [1, 3, 10]:
    kk = gauss_kernel(sg)
    print(f"sigma={sg:2d}: ||f*g||_2 = {np.linalg.norm(conv_fft(img, kk)):.2f} <= ||f||_2 ||g||_1 = {np.linalg.norm(img) * np.abs(kk).sum():.2f}")

A digital image is a function on a grid (here a torus, since the FFT convolution is periodic), and blurring is convolution with a discrete Gaussian of total mass $1$, an approximate identity in reverse: the larger $\sigma$, the more the image is averaged. Direct summation costs $O(n^2s^2)$ operations for an $s\times s$ kernel, the FFT $O(n^2\log n)$ whatever the kernel; the reason the FFT turns convolution into a product is the Fourier theory of finite abelian groups (Volume 7, Chapter 7). Unsharp masking adds back a multiple of the detail $f-f*G_t$, which is what the "sharpen" button of photo editors does; the Laplacian kernel, a discrete second derivative, vanishes on flat regions and lights up at edges. Young's inequality holds with room to spare: averaging decreases the $L^2$ norm, and more so for wider kernels.

### P9.3

In [ ]:
import numpy as np
x = np.linspace(-3, 3, 60001); dx = x[1] - x[0]
f = (np.abs(x) <= 1).astype(float)
kernels = {"Gauss": lambda z: np.exp(-z ** 2 / 2) / np.sqrt(2 * np.pi),
           "Poisson": lambda z: 1 / (np.pi * (1 + z ** 2)),
           "box": lambda z: (np.abs(z) <= 0.5).astype(float)}
for name, phi in kernels.items():
    for t in [0.3, 0.1, 0.03, 0.01]:
        k = phi(x / t) / t * dx
        g = np.convolve(f, k, mode="same")
        mid = len(x) // 2 + int(round(1 / dx))                     # index of x = 1
        print(f"{name:7s} t={t:5.2f}: L1 error {np.sum(np.abs(g - f)) * dx:.4f}   sup error {np.max(np.abs(g - f)):.3f}   value at the jump {g[mid]:.3f}")

In $L^1$ the error tends to $0$ for every kernel, roughly proportionally to $t$ for the Gauss and box kernels and like $t\ln\frac1t$ for the Poisson kernel, whose heavy tails $\frac1{\pi z^2}$ leak mass far from the jumps (the computation is truncated to $[-3,3]$, which slightly underestimates its error for large $t$). The sup error stays near $\frac12$: a uniform limit of continuous functions is continuous, so no approximate identity can converge uniformly to a step. At the jump the value is about $\frac12$, the average of the two sides, for every symmetric kernel. Theorem 9.10 gives exactly this: convergence in $L^p$ for $p<\infty$, and pointwise convergence at points of continuity only.

### P9.4

In [ ]:
import numpy as np
for p in [1.5, 2, 4]:
    q = p / (p - 1)
    for N in [1e2, 1e4, 1e8, 1e16]:
        # f = x^(-1/p) on [1,N]: F = q (x^(-1/p) - 1/x) on [1,N], F = q (N^(1/q) - 1)/x beyond
        u = np.linspace(0, np.log(N), 200001); xs = np.exp(u)                # integrate in log scale
        F = q * (xs ** (-1 / p) - 1 / xs)
        inner = np.trapezoid(F ** p * xs, u)
        tail = (q * (N ** (1 / q) - 1)) ** p * N ** (1 - p) / (p - 1)
        ratio = ((inner + tail) / np.log(N)) ** (1 / p)
        print(f"p={p}: N=1e{int(np.log10(N)):2d}  ||F||_p/||f||_p = {ratio:.4f}   p/(p-1) = {q:.4f}")

The ratio increases towards $\frac p{p-1}$, but only logarithmically in $N$: the norm $\|f\|_p^p=\ln N$ grows slowly, and the constant losses near $x=1$ and in the tail are bounded. Extremal functions for Hardy's inequality do not exist; near-extremal ones look like $x^{-1/p}$, which just fails to be in $L^p$ at both ends, the same borderline as $\frac1{\sqrt x}$ for $p=2$ in the chapter.